# Train SLM Router - Qwen3-1.7B Multi-task Fine-tuning (Kaggle)

**Task:** Fine-tune Qwen3-1.7B for dual-output routing:
- Primary: Intent classification + scope detection (15 intents x 3 scopes)
- Secondary (multi-task): Contextual query rewriting with conversation context

**Output schema:**
```json
// Turn 1: {"intent": "current_weather", "scope": "district", "confidence": 0.95}
// Turn 2+: {"intent": "daily_forecast", "scope": "district", "confidence": 0.91,
//           "rewritten_query": "Du bao ngay mai quan Cau Giay?"}
```

## Setup (bat buoc truoc khi chay)
1. Chay `python scripts/router/generate_rewrite_data.py --use-llm` de tao `multitask_train.jsonl` va `multitask_val.jsonl`
2. Upload ca 2 file len Kaggle Datasets
2. Settings -> Add Data -> chon dataset vua upload
3. Settings -> Accelerator -> GPU T4 x2 hoac P100
4. Settings -> Internet -> On
5. Settings -> Secrets -> them `HF_TOKEN` (optional, neu muon push Hub)


In [ ]:
# Cell 1: Install packages
# unsloth[kaggle-new] = build toi uu cho Kaggle T4/P100
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "unsloth[kaggle-new]", "trl>=0.16",
], check=True)
print("Packages installed")

In [ ]:
# Cell 2: Imports
import json, os
from pathlib import Path
from collections import Counter
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
# TRL >= 0.17 removed DataCollatorForCompletionOnlyLM; replaced by train_on_responses_only
try:
    from trl import train_on_responses_only
    _HAS_TORO = True
except ImportError:
    _HAS_TORO = False  # fallback: train on full sequence (slightly less efficient, still correct)
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"train_on_responses_only available: {_HAS_TORO}")

In [19]:
# Cell 3: Configuration

# --- Model ---
BASE_MODEL = "unsloth/Qwen3-1.7B-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/outputs"
GGUF_DIR   = "/kaggle/working/gguf"

# --- Data ---
DATA_DIR       = Path("/kaggle/input/datasets/tuantdt/chatbot-hanoi-weather")
TRAIN_FILE     = DATA_DIR / "multitask_train.jsonl"  # unified: routing + rewrite
VAL_FILE       = DATA_DIR / "multitask_val.jsonl"    # converted val set

# --- LoRA ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0

# --- Training ---
MAX_SEQ_LENGTH = 1024   # Tang tu 768 cho truong rewritten_query
EPOCHS         = 5
BATCH_SIZE     = 4      # T4 16GB handles 4 with 1.7B
GRAD_ACCUM     = 8      # Effective batch = 32
LR             = 2e-4
WARMUP_RATIO   = 0.05

# --- Intents / Scopes (must match app/agent/router/config.py) ---
VALID_INTENTS = [
    "current_weather", "hourly_forecast", "daily_forecast", "weather_overview",
    "rain_query", "temperature_query", "wind_query", "humidity_fog_query",
    "historical_weather", "location_comparison", "activity_weather",
    "expert_weather_param", "weather_alert", "seasonal_context", "smalltalk_weather",
]
VALID_SCOPES = ["city", "district", "ward"]

# --- System prompt (must match ROUTER_SYSTEM_PROMPT in config.py) ---
SYSTEM_PROMPT = '''Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)
- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)
- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (không hỏi thông số cụ thể)
- rain_query: hỏi CÓ MƯA KHÔNG, mưa bao lâu, xác suất mưa, mưa lúc nào tạnh
- temperature_query: hỏi CỤ THỂ VỀ NHIỆT ĐỘ (bao nhiêu độ, nóng/lạnh thế nào)
- wind_query: hỏi CỤ THỂ VỀ GIÓ (gió mạnh không, hướng gió, tốc độ gió)
- humidity_fog_query: hỏi về ĐỘ ẨM, SƯƠNG MÙ, sương muối
- historical_weather: thời tiết NGÀY/TUẦN TRƯỚC, dữ liệu QUÁ KHỨ
- location_comparison: SO SÁNH thời tiết giữa các quận/phường/địa điểm
- activity_weather: thời tiết có PHÙ HỢP ĐỂ LÀM hoạt động X không (chạy bộ, picnic, du lịch)
- expert_weather_param: thông số KỸ THUẬT chuyên sâu (áp suất, tia UV, điểm sương, tầm nhìn)
- weather_alert: CẢNH BÁO, nguy hiểm, bão, ngập, thay đổi đột ngột
- seasonal_context: SO SÁNH với hôm qua/tuần trước, xu hướng, bất thường theo MÙA
- smalltalk_weather: chào hỏi, ngoài phạm vi (không phải Hà Nội), câu hỏi không liên quan thời tiết

## Scopes:
- city: toàn Hà Nội, hoặc KHÔNG NÓI RÕ địa điểm
- district: nhắc tên QUẬN/HUYỆN (Ba Đình, Cầu Giấy...) hoặc ĐỊA ĐIỂM NỔI TIẾNG thuộc quận (Hồ Gươm→Hoàn Kiếm, Lăng Bác→Ba Đình, Nội Bài→Sóc Sơn...)
- ward: nhắc tên PHƯỜNG/XÃ (Phường Dịch Vọng Hậu, Xã Tiên Dương...)

## Multi-task output (khi có context từ lượt trước):
Nếu được cung cấp context (location/intent từ lượt trước) VÀ câu hỏi hiện tại thiếu địa điểm hoặc dùng đại từ (ở đó, thế còn, còn ..., vậy ...), hãy điền thêm trường "rewritten_query" với câu hỏi đầy đủ ngữ cảnh.

## Output format:
Khi không có context hoặc câu hỏi đầy đủ:
{"intent": "...", "scope": "...", "confidence": 0.92}

Khi có context và cần rewrite:
{"intent": "...", "scope": "...", "confidence": 0.91, "rewritten_query": "Dự báo thời tiết ngày mai ở quận Cầu Giấy như thế nào?"}'''

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

print(f"Base model:      {BASE_MODEL}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Train file:      {TRAIN_FILE.name}")
print(f"Val file:        {VAL_FILE.name}")
print(f"Max seq length:  {MAX_SEQ_LENGTH}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

Base model:      unsloth/Qwen3-1.7B-bnb-4bit
Data dir exists: True
Train file:      multitask_train.jsonl
Val file:        multitask_val.jsonl
Max seq length:  1024
Effective batch: 32


In [14]:
# Cell 4: Load base model (Qwen3-1.7B 4-bit QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # auto: bfloat16 on A100, float16 on T4/P100
    load_in_4bit=True,
)
print(f"Model: {type(model).__name__}")
print(f"Vocab: {tokenizer.vocab_size}")
print(f"Dtype: {next(model.parameters()).dtype}")

==((====))==  Unsloth 2025.10.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model: Qwen3ForCausalLM
Vocab: 151643
Dtype: torch.float16


In [15]:
# Cell 5: Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Trainable: 17,432,576 / 1,033,364,480 (1.69%)


In [16]:
# Cell 6: Chat template
# Qwen3 uses ChatML (same format as Qwen2.5).
# enable_thinking=False: disable <think> blocks for fast JSON router output.
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

sample = [{"role": "system", "content": "Test"}, {"role": "user", "content": "Hello"}]
try:
    preview = tokenizer.apply_chat_template(
        sample, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
except TypeError:
    preview = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
print("Chat template preview:")
print(preview[:200])

Chat template preview:
<|im_start|>system
Test<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant



In [20]:
# Cell 7: Load and validate data
# multitask_train.jsonl and multitask_val.jsonl use unified input/output format.
# Generated by: python scripts/router/generate_rewrite_data.py --use-llm

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"  Line {i} error: {e}")
    return records

def validate_record(rec, idx):
    """Validate input/output format record."""
    out = rec.get("output", {})
    if isinstance(out, str):
        try: out = json.loads(out)
        except: return False
    return (out.get("intent") in VALID_INTENTS and
            out.get("scope")  in VALID_SCOPES  and
            bool(rec.get("input")))

# Load training data
print(f"Loading {TRAIN_FILE}...")
all_records = load_jsonl(TRAIN_FILE)
valid_records = [r for i, r in enumerate(all_records) if validate_record(r, i)]
multitask_count = sum(1 for r in valid_records if r.get("output", {}).get("rewritten_query"))
print(f"  Total: {len(valid_records)} / {len(all_records)} valid")
print(f"  Routing-only: {len(valid_records) - multitask_count}")
print(f"  With rewrite: {multitask_count}")

# Load validation data
print(f"Loading {VAL_FILE}...")
val_raw = load_jsonl(VAL_FILE)
val_records = [r for i, r in enumerate(val_raw) if validate_record(r, i)]
print(f"  Val: {len(val_records)}")

# Intent distribution
intent_counts = Counter()
for r in valid_records:
    out = r.get("output", {})
    if isinstance(out, str): out = json.loads(out)
    intent_counts[out.get("intent", "?")] += 1

print("\nIntent distribution:")
for k, v in sorted(intent_counts.items(), key=lambda x: -x[1]):
    pct = v / len(valid_records) * 100
    print(f"  {k:<30} {v:>4} ({pct:4.1f}%)")

Loading /kaggle/input/datasets/tuantdt/chatbot-hanoi-weather/multitask_train.jsonl...
  Total: 1983 / 1983 valid
  Routing-only: 1340
  With rewrite: 643
Loading /kaggle/input/datasets/tuantdt/chatbot-hanoi-weather/multitask_val.jsonl...
  Val: 299

Intent distribution:
  daily_forecast                  169 ( 8.5%)
  rain_query                      166 ( 8.4%)
  weather_alert                   146 ( 7.4%)
  current_weather                 142 ( 7.2%)
  wind_query                      139 ( 7.0%)
  activity_weather                137 ( 6.9%)
  humidity_fog_query              131 ( 6.6%)
  temperature_query               131 ( 6.6%)
  expert_weather_param            130 ( 6.6%)
  seasonal_context                129 ( 6.5%)
  weather_overview                123 ( 6.2%)
  hourly_forecast                 116 ( 5.8%)
  location_comparison             110 ( 5.5%)
  historical_weather              108 ( 5.4%)
  smalltalk_weather               106 ( 5.3%)


In [ ]:
# Cell 8: Format data for SFT (completion-only training)
# Loss is computed only on the assistant turn.

def format_record(rec):
    user_msg = str(rec.get("input", "")).strip()
    # ── Inject conversation context so the model can learn rewriting ──
    ctx = rec.get("context")
    if ctx:
        ctx_str = json.dumps(ctx, ensure_ascii=False, separators=(",", ":"))
        user_msg = f"[CONTEXT: {ctx_str}]\n{user_msg}"
    out = rec.get("output", {})
    if isinstance(out, str): out = json.loads(out)

    output_dict = {
        "intent":     out["intent"],
        "scope":      out["scope"],
        "confidence": round(float(out.get("confidence", 0.9)), 2),
    }
    rw = out.get("rewritten_query")
    if rw and str(rw).strip():
        output_dict["rewritten_query"] = str(rw).strip()

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": json.dumps(output_dict, ensure_ascii=False)},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )


print("Formatting training data...")
train_texts = [format_record(r) for r in valid_records]
val_texts   = [format_record(r) for r in val_records]

print("\nSample:")
print("-" * 60)
print(train_texts[0])
print("-" * 60)

lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f"Lengths (first 200): min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
over = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"Over limit: {over}/200")

In [ ]:
# Cell 9: HuggingFace datasets
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset   = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# Cell 10: Completion-only training setup
# TRL >= 0.17: DataCollatorForCompletionOnlyLM removed.
# Loss is restricted to assistant turn via train_on_responses_only (applied after trainer init).
# This cell is intentionally a no-op placeholder — see Cell 12 for actual setup.
print("Completion-only training will be configured in Cell 12 via train_on_responses_only.")

In [ ]:
# Cell 11: Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    report_to="none",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
)
prec = "bf16" if training_args.bf16 else "fp16"
print(f"Precision: {prec} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps/epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

In [ ]:
# Cell 12: SFT Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,   # TRL >= 0.16: was tokenizer=
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,
)

# Apply completion-only loss masking (TRL >= 0.17 API)
if _HAS_TORO:
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("Completion-only training enabled via train_on_responses_only")
else:
    print("Fallback: training on full sequence (DataCollatorForCompletionOnlyLM unavailable)")

if torch.cuda.is_available():
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total_v  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM reserved: {reserved:.1f} GB / {total_v:.1f} GB")

In [ ]:
# Cell 13: TRAIN
print("=" * 60)
print(f"Model:  {BASE_MODEL}")
print(f"LoRA:   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:  {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Epochs: {EPOCHS}")
print("=" * 60)

train_result = trainer.train()

print(f"Train loss:  {train_result.training_loss:.4f}")
print(f"Runtime:     {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/s:   {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# Cell 14: Save LoRA adapter
adapter_dir = Path(OUTPUT_DIR) / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter: {adapter_dir}")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Cell 14b: Full evaluation on multitask_val.jsonl (~15 min on T4)
# Runs inference on all 299 val examples → 4 metrics embedded in output for offline analysis.
print("=" * 60)
print("Running full evaluation on multitask_val.jsonl...")
print("=" * 60)

FastLanguageModel.for_inference(model)

val_examples = [
    json.loads(l)
    for l in Path(VAL_FILE).read_text(encoding="utf-8").splitlines()
    if l.strip()
]
print(f"Loaded {len(val_examples)} val examples")

r_routing, r_rewrite, r_norewrite = [], [], []
entity_hits = 0
rw_present  = 0
norewrite_ok = 0

for i, ex in enumerate(val_examples):
    ctx      = ex.get("context")
    gt       = ex["output"]
    gt_rw    = gt.get("rewritten_query")
    user_msg = str(ex["input"])

    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",", ":"))
        user_msg = f"[CONTEXT: {ctx_str}]\n{user_msg}"

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    try:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=150, do_sample=False, use_cache=True)
    resp = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    try:
        pred = json.loads(resp)
    except Exception:
        pred = {}

    intent_ok = (pred.get("intent") == gt.get("intent"))
    scope_ok  = (pred.get("scope")  == gt.get("scope"))
    ok_both   = intent_ok and scope_ok
    pred_rw   = pred.get("rewritten_query") or None

    if ctx is None:
        # Routing-only example
        r_routing.append(ok_both)
    elif gt_rw:
        # With-rewrite example: check routing + presence + entity
        r_rewrite.append(ok_both)
        if pred_rw:
            rw_present += 1
            loc = ctx.get("location", "")
            if loc and loc.lower() in pred_rw.lower():
                entity_hits += 1
    else:
        # No-rewrite example: rewritten_query must be ABSENT
        r_norewrite.append(ok_both)
        if pred_rw is None:
            norewrite_ok += 1

    if (i + 1) % 50 == 0:
        print(f"  Progress: {i+1}/{len(val_examples)}")

# ── Report ───────────────────────────────────────────────────
def pct(a, b):
    return f"{a}/{b} = {a/b*100:.1f}%" if b else "N/A"

n_rw = len(r_rewrite)  or 1
n_nr = len(r_norewrite) or 1

print("\n" + "=" * 60)
print("FULL EVAL RESULTS — multitask_val.jsonl")
print("=" * 60)
print(f"Routing accuracy    : {pct(sum(r_routing),  len(r_routing))}   (target ≥ 90%)")
print(f"Rewrite routing acc : {pct(sum(r_rewrite),  n_rw)}   (target ≥ 85%)")
print(f"No-rewrite routing  : {pct(sum(r_norewrite),n_nr)}   (target ≥ 80%)")
print(f"--- rewrite quality ---")
print(f"Rewrite presence    : {pct(rw_present,  n_rw)}   (model outputs rewritten_query)")
print(f"Entity coverage     : {pct(entity_hits, n_rw)}   (location from ctx in rewrite)")
print(f"No-rewrite (absent) : {pct(norewrite_ok,n_nr)}   (model correctly omits field)")
print("=" * 60)
PASS_ROUTING = (sum(r_routing) / len(r_routing)) >= 0.90 if r_routing else False
PASS_ENTITY  = (entity_hits / n_rw) >= 0.70
print(f"\nPass: routing≥90% → {'✅' if PASS_ROUTING else '❌'}  |  entity≥70% → {'✅' if PASS_ENTITY else '❌'}")
if PASS_ROUTING and PASS_ENTITY:
    print("→ Deploy to Ollama ✅")
else:
    print("→ Review errors before deploy ⚠️")


In [ ]:
# Cell 15: Inference test (v2 — shows full rewritten_query, checks entity coverage)
FastLanguageModel.for_inference(model)

TEST_CASES = [
    # (description, user_msg_with_optional_context, expected_intent, expected_entity_or_None)
    ("Routing",
     "Bây giờ thời tiết Cầu Giấy thế nào?",
     "current_weather", None),

    ("Forecast",
     "Cuối tuần Hà Nội thế nào?",
     "daily_forecast", None),

    ("Rewrite — location carry",
     '[CONTEXT: {"location":"Cầu Giấy","intent":"current_weather","turn":1}]\nCòn ngày mai?',
     "daily_forecast", "Cầu Giấy"),

    ("Rewrite — pronoun",
     '[CONTEXT: {"location":"Hoàng Mai","intent":"rain_query","turn":1}]\nỞ đó có gió không?',
     "wind_query", "Hoàng Mai"),

    ("No-rewrite — explicit location",
     '[CONTEXT: {"location":"Cầu Giấy","intent":"current_weather","turn":1}]\nThời tiết Đống Đa ngày mai?',
     "daily_forecast", None),   # explicit location → no rewritten_query expected

    ("Activity",
     "Sáng mai đi chạy bộ được không?",
     "activity_weather", None),
]

print("=" * 70)
print("INFERENCE TEST (v2)")
print("=" * 70)
all_ok = True
for desc, user_msg, exp_intent, exp_entity in TEST_CASES:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}]
    try:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    import torch as _t
    with _t.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=150,
            temperature=1.0, do_sample=False, use_cache=True)
    resp = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Parse
    ok_intent = False; ok_entity = True; pred_intent = "?"; pred_rw = None
    try:
        parsed     = json.loads(resp)
        pred_intent = parsed.get("intent", "?")
        pred_rw     = parsed.get("rewritten_query")
        ok_intent   = (pred_intent == exp_intent)
        if exp_entity:                          # rewrite expected → entity must appear
            ok_entity = bool(pred_rw) and exp_entity.lower() in pred_rw.lower()
        else:                                   # no-rewrite case: rewritten_query should be absent
            ok_entity = (pred_rw is None)
    except Exception:
        ok_intent = False

    ok = ok_intent and ok_entity
    if not ok:
        all_ok = False
    status = "✅" if ok else "❌"

    print(f"\n{status} [{desc}]")
    print(f"   Input:   {user_msg[:80]}")
    print(f"   Intent:  {pred_intent!r}  (expected {exp_intent!r})  {'✓' if ok_intent else '✗'}")
    if exp_entity:
        print(f"   Entity:  {exp_entity!r} in rewrite → {'✓' if ok_entity else '✗'}")
        print(f"   Rewrite: {pred_rw}")
    else:
        print(f"   No-rewrite check → rewritten_query absent: {'✓' if ok_entity else '✗'}")

print(f"\n{'='*70}")
print(f"All passed: {all_ok}  ({sum(1 for _,_,e,_ in TEST_CASES if True)}/6 cases)")


In [ ]:
# Cell 16: Export GGUF (with manual llama.cpp build for Kaggle)
# Unsloth's auto-build fails because llama.cpp switched to CMake.
# We build llama.cpp manually first, then let unsloth use it.

import subprocess, shutil

LLAMA_CPP_DIR = "/tmp/llama.cpp"

# Step 1: Clone and build llama.cpp with CMake
if not Path(LLAMA_CPP_DIR).exists():
    print("Cloning llama.cpp...")
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggml-org/llama.cpp.git",
                    LLAMA_CPP_DIR], check=True)

print("Building llama.cpp with CMake...")
build_dir = f"{LLAMA_CPP_DIR}/build"
subprocess.run(["cmake", LLAMA_CPP_DIR, "-B", build_dir,
                "-DBUILD_SHARED_LIBS=OFF", "-DGGML_CUDA=OFF"],
               check=True)
subprocess.run(["cmake", "--build", build_dir, "--config", "Release",
                "-j", str(os.cpu_count() or 4),
                "--target", "llama-quantize", "--target", "llama-gguf-hash"],
               check=True)

# Step 2: Save merged 16-bit model
MERGED_DIR = "/tmp/merged_model"
print("Saving merged 16-bit model...")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# Step 3: Convert HF -> GGUF f16
print("Converting HF -> GGUF f16...")
converter = f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py"
f16_gguf = f"/tmp/model-f16.gguf"
subprocess.run([sys.executable, converter, MERGED_DIR,
                "--outfile", f16_gguf, "--outtype", "f16"], check=True)

# Step 4: Quantize to q8_0 and q4_k_m
quantizer = f"{build_dir}/bin/llama-quantize"
Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

for qtype in ["q8_0", "q4_k_m"]:
    out_path = f"{GGUF_DIR}/{qtype}/unsloth.{qtype.upper()}.gguf"
    Path(f"{GGUF_DIR}/{qtype}").mkdir(parents=True, exist_ok=True)
    print(f"Quantizing {qtype}...")
    subprocess.run([quantizer, f16_gguf, out_path, qtype], check=True)

# Cleanup
os.remove(f16_gguf)
shutil.rmtree(MERGED_DIR, ignore_errors=True)

print("\nGGUF files:")
for f in sorted(Path(GGUF_DIR).rglob("*.gguf")):
    print(f"  {f.relative_to(GGUF_DIR)}  ({f.stat().st_size/1e9:.2f} GB)")

In [ ]:
# Cell 17: Generate Ollama Modelfile
# GGUF filename from Cell 16: unsloth.Q8_0.gguf
gguf_name = "unsloth.Q8_0.gguf"

modelfile = f"""FROM ./{gguf_name}

SYSTEM \"{SYSTEM_PROMPT}\"

PARAMETER temperature 0
PARAMETER num_predict 128
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
"""

mf = Path(GGUF_DIR) / "q8_0" / "Modelfile"
mf.write_text(modelfile, encoding="utf-8")
print(f"Modelfile: {mf}")
print("\nDeploy on Ollama:")
print(f"  1. Download q8_0/ folder from Kaggle Output tab")
print(f"  2. cd into the folder containing {gguf_name} and Modelfile")
print("  3. ollama create hanoi-weather-router -f Modelfile")
print("  4. Test: ollama run hanoi-weather-router")
print("\n.env: OLLAMA_MODEL=hanoi-weather-router  USE_SLM_ROUTER=true")

In [ ]:
# Cell 18: (Optional) Push to HuggingFace Hub
PUSH_TO_HUB  = True
HUB_MODEL_ID = "daredevil467/qwen3-1.7b-hanoi-weather-router-v1"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = os.getenv("HF_TOKEN", "")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        model.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        tokenizer.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        print(f"Pushed: https://huggingface.co/{HUB_MODEL_ID}")
    else:
        print("HF_TOKEN not in Kaggle Secrets")
else:
    print("Hub push skipped (set PUSH_TO_HUB=True to enable)")

In [ ]:
# Cell 19: Summary
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Base:   {BASE_MODEL}")
print(f"LoRA:   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:  {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Loss:   {train_result.training_loss:.4f}")
print()
print("Output files (Kaggle Output tab):")
for root, dirs, files in os.walk("/kaggle/working"):
    for fname in sorted(files):
        fp = os.path.join(root, fname)
        sz = os.path.getsize(fp) / 1e6
        if sz > 0.1:
            rel = os.path.relpath(fp, "/kaggle/working")
            print(f"  {rel:<55} {sz:7.1f} MB")